<a href="https://colab.research.google.com/github/codeKrantz/Iowa-State-AI-2010/blob/main/Titanic_Machine_Learning_from_Disaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

In [ ]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
train_data.head()

In [ ]:
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
test_data.head()

In [ ]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

In [ ]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

## Exploratory Data Analysis

Before trusting a model, look at the data. The sections below chart what
actually drove survival on the Titanic: **sex above everything**, then
passenger class, age, fare and family size.

In [ ]:
# ---- Setup: imports and a consistent chart style ----------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

# Palette: one colour per meaning, used the same way in every chart.
# Blue = survived / survival rate. Orange = did not survive. Nothing else.
SURVIVED = "#2a78d6"   # blue   - survived
DIED     = "#eb6834"   # orange - did not survive
SURFACE  = "#fcfcfb"
INK      = "#0b0b0b"
SECOND   = "#52514e"
MUTED    = "#898781"
GRID     = "#e1e0d9"
BASELINE = "#c3c2b7"

# A single blue ramp for "how much" (heatmaps), blue->grey->red for correlation.
BLUES = LinearSegmentedColormap.from_list("blues", ["#eaf2fd", "#9ec5f4", "#3987e5", "#1c5cab", "#0d366b"])
DIVERGING = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "font.size": 10.5,
    "axes.edgecolor": BASELINE, "axes.linewidth": 0.8, "axes.labelcolor": SECOND,
    "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": SECOND, "ytick.labelcolor": SECOND,
    "xtick.major.size": 0, "ytick.major.size": 0,
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "legend.frameon": False, "legend.fontsize": 9.5,
})

def tidy(ax, grid_axis="y"):
    """Recessive grid behind the data, no clutter."""
    ax.grid(axis=grid_axis, zorder=0)
    ax.set_axisbelow(True)
    return ax

def pct_labels(ax, bars, values, color=INK, size=10):
    """Direct-label each bar with its percentage."""
    ax.bar_label(bars, labels=[f"{v:.1%}" for v in values], padding=4,
                 color=color, fontsize=size, fontweight="bold")

def cell_gaps(ax, n_rows, n_cols):
    """Hairline surface gaps between heatmap cells, so blocks don't fuse."""
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="minor", length=0)

print(f"train: {train_data.shape[0]} rows x {train_data.shape[1]} columns")
print(f"test:  {test_data.shape[0]} rows x {test_data.shape[1]} columns")
print(f"\nOverall survival rate in the training set: {train_data['Survived'].mean():.1%}")

In [ ]:
# ---- Chart 1: where the data is missing -------------------------------------
missing = (train_data.isna().mean() * 100).sort_values(ascending=True)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(9, 3.2))
bars = ax.barh(missing.index, missing.values, color=SURVIVED,
               height=0.62, edgecolor=SURFACE, linewidth=1.5, zorder=3)
ax.bar_label(bars, labels=[f"{v:.1f}%" for v in missing.values],
             padding=5, color=INK, fontsize=10, fontweight="bold")
ax.set_xlim(0, 100)
ax.set_xlabel("% of rows missing")
ax.set_title("Missing data: Cabin is unusable, Age needs filling")
tidy(ax, grid_axis="x")
plt.tight_layout()
plt.show()

print("Cabin is missing for most passengers, so it is dropped.")
print("Age is missing for ~20% and is worth imputing rather than discarding.")

### The headline: sex was the single strongest signal

"Women and children first" is not a story about the Titanic — it is
measurable in the data, and it is by far the largest effect in the dataset.

In [ ]:
# ---- Chart 2: THE gender chart ----------------------------------------------
by_sex = train_data.groupby("Sex")["Survived"].agg(["mean", "sum", "count"])
by_sex = by_sex.reindex(["female", "male"])
overall = train_data["Survived"].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4))

# Left: survival RATE by sex
labels = [f"Women\nn={int(by_sex.loc['female','count'])}",
          f"Men\nn={int(by_sex.loc['male','count'])}"]
bars = ax1.bar(labels, by_sex["mean"], color=SURVIVED, width=0.45,
               edgecolor=SURFACE, linewidth=2, zorder=3)
pct_labels(ax1, bars, by_sex["mean"], size=16)
ax1.axhline(overall, color=MUTED, linewidth=1, linestyle="-", zorder=2)
ax1.text(1.45, overall + 0.025, f"all passengers {overall:.1%}",
         color=MUTED, fontsize=9, ha="right")
ax1.set_ylim(0, 0.92)
ax1.set_ylabel("Survival rate")
ax1.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax1.set_title("Women survived at ~4x the rate of men")
tidy(ax1)

# Right: the same fact in absolute numbers
x = np.arange(2)
w = 0.38
b1 = ax2.bar(x - w/2, by_sex["sum"], w, label="Survived",
             color=SURVIVED, edgecolor=SURFACE, linewidth=2, zorder=3)
b2 = ax2.bar(x + w/2, by_sex["count"] - by_sex["sum"], w, label="Did not survive",
             color=DIED, edgecolor=SURFACE, linewidth=2, zorder=3)
ax2.bar_label(b1, padding=4, color=INK, fontsize=10, fontweight="bold")
ax2.bar_label(b2, padding=4, color=INK, fontsize=10, fontweight="bold")
ax2.set_xticks(x, ["Women", "Men"])
ax2.set_ylim(0, 540)
ax2.set_ylabel("Passengers")
ax2.set_title("Most men aboard died; most women did not")
ax2.legend(loc="upper left")
tidy(ax2)

plt.tight_layout()
plt.show()

ratio = by_sex.loc["female", "mean"] / by_sex.loc["male", "mean"]
print(f"Women:  {by_sex.loc['female','mean']:.1%} survived "
      f"({int(by_sex.loc['female','sum'])} of {int(by_sex.loc['female','count'])})")
print(f"Men:    {by_sex.loc['male','mean']:.1%} survived "
      f"({int(by_sex.loc['male','sum'])} of {int(by_sex.loc['male','count'])})")
print(f"\n=> A woman aboard was {ratio:.1f}x more likely to survive than a man.")
print("   Predicting 'every woman lives, every man dies' alone scores ~0.766 on Kaggle.")

In [ ]:
# ---- Chart 3: class, and how it interacts with sex --------------------------
by_class = train_data.groupby("Pclass")["Survived"].mean()
grid = train_data.pivot_table(index="Sex", columns="Pclass",
                              values="Survived", aggfunc="mean")
grid = grid.reindex(["female", "male"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2),
                               gridspec_kw={"width_ratios": [1, 1.15]})

bars = ax1.bar([f"{c}{'st' if c==1 else 'nd' if c==2 else 'rd'} class" for c in by_class.index],
               by_class.values, color=SURVIVED, width=0.55,
               edgecolor=SURFACE, linewidth=2, zorder=3)
pct_labels(ax1, bars, by_class.values, size=12)
ax1.set_ylim(0, 0.78)
ax1.set_ylabel("Survival rate")
ax1.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax1.set_title("Survival fell with every class")
tidy(ax1)

im = ax2.imshow(grid.values, cmap=BLUES, vmin=0, vmax=1, aspect="auto")
ax2.set_xticks(range(3), ["1st class", "2nd class", "3rd class"])
ax2.set_yticks(range(2), ["Women", "Men"])
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        v = grid.values[i, j]
        ax2.text(j, i, f"{v:.0%}", ha="center", va="center",
                 color="#ffffff" if v > 0.55 else INK, fontsize=14, fontweight="bold")
ax2.set_title("Sex x class: from 97% down to 14%")
ax2.grid(False)
cell_gaps(ax2, 2, 3)
for s in ax2.spines.values():
    s.set_visible(False)
cb = fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.03)
cb.set_label("Survival rate", color=SECOND, fontsize=9)
cb.ax.tick_params(colors=MUTED, labelcolor=SECOND, length=0)
cb.ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
cb.outline.set_visible(False)

plt.tight_layout()
plt.show()

print("Class matters, but it never overturns sex:")
print(f"  a woman in 3rd class ({grid.loc['female', 3]:.0%}) still outlived "
      f"a man in 1st class ({grid.loc['male', 1]:.0%}).")

In [ ]:
# ---- Chart 4: age -----------------------------------------------------------
age = train_data.dropna(subset=["Age"])
bins = np.arange(0, 85, 5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))

ax1.hist([age.loc[age.Survived == 1, "Age"], age.loc[age.Survived == 0, "Age"]],
         bins=bins, color=[SURVIVED, DIED], label=["Survived", "Did not survive"],
         edgecolor=SURFACE, linewidth=0.8, zorder=3)
ax1.set_xlabel("Age")
ax1.set_ylabel("Passengers")
ax1.set_title("Young children are the visible exception")
ax1.legend(loc="upper right")
tidy(ax1)

band_edges = [0, 12, 18, 30, 45, 60, 100]
band_names = ["0-12", "13-17", "18-29", "30-44", "45-59", "60+"]
bands = pd.cut(age["Age"], bins=band_edges, labels=band_names, right=False)
by_band = age.groupby(bands, observed=False)["Survived"].agg(["mean", "count"])

bars = ax2.bar(range(len(by_band)), by_band["mean"], color=SURVIVED,
               width=0.6, edgecolor=SURFACE, linewidth=2, zorder=3)
pct_labels(ax2, bars, by_band["mean"], size=10)
ax2.set_xticks(range(len(by_band)),
               [f"{b}\nn={n}" for b, n in zip(by_band.index.astype(str), by_band["count"])])
ax2.axhline(overall, color=MUTED, linewidth=1, zorder=2)
ax2.text(len(by_band) - 0.45, overall + 0.018, f"average {overall:.1%}",
         color=MUTED, fontsize=9, ha="right")
ax2.set_ylim(0, 0.72)
ax2.set_ylabel("Survival rate")
ax2.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax2.set_title("Children did best, the oldest worst")
tidy(ax2)

plt.tight_layout()
plt.show()

kids = age[age.Age < 13]["Survived"].mean()
old = age[age.Age >= 60]["Survived"].mean()
print(f"Children under 13: {kids:.1%} survived, against {overall:.1%} overall.")
print(f"Passengers 60 and over: {old:.1%}.")
print("Age is weak on its own but useful once it is combined with sex.")

In [ ]:
# ---- Chart 5: fare (and why it mostly restates class) -----------------------
fare = train_data.copy()
fare["FareQ"] = pd.qcut(fare["Fare"], 4, labels=["cheapest 25%", "25-50%", "50-75%", "priciest 25%"])
by_fare = fare.groupby("FareQ", observed=False)["Survived"].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))

bars = ax1.bar(by_fare.index.astype(str), by_fare.values, color=SURVIVED,
               width=0.6, edgecolor=SURFACE, linewidth=2, zorder=3)
pct_labels(ax1, bars, by_fare.values, size=11)
ax1.set_ylim(0, 0.72)
ax1.set_ylabel("Survival rate")
ax1.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax1.set_xlabel("Ticket fare quartile")
ax1.set_title("The pricier the ticket, the better the odds")
tidy(ax1)

data = [train_data.loc[train_data.Pclass == c, "Fare"].dropna() for c in [1, 2, 3]]
bp = ax2.boxplot(data, patch_artist=True, widths=0.45,
                 medianprops=dict(color=INK, linewidth=1.6),
                 whiskerprops=dict(color=BASELINE, linewidth=1),
                 capprops=dict(color=BASELINE, linewidth=1),
                 flierprops=dict(marker="o", markersize=3.5,
                                 markerfacecolor=MUTED, markeredgecolor="none", alpha=0.5))
for patch in bp["boxes"]:
    patch.set_facecolor("#cde2fb")
    patch.set_edgecolor(SURVIVED)
    patch.set_linewidth(1.4)
ax2.set_xticks([1, 2, 3], ["1st class", "2nd class", "3rd class"])
ax2.set_ylim(0, 300)
ax2.set_ylabel("Fare (£)")
ax2.set_title("Fare is largely a restatement of class")
tidy(ax2)

plt.tight_layout()
plt.show()

print("Fare and Pclass carry much of the same information "
      f"(correlation {train_data['Fare'].corr(train_data['Pclass']):.2f}),")
print("so adding Fare on top of Pclass buys less than it first appears.")

In [ ]:
# ---- Chart 6: travelling alone vs. with family ------------------------------
fam = train_data.copy()
fam["FamilySize"] = fam["SibSp"] + fam["Parch"] + 1
by_fam = fam.groupby("FamilySize")["Survived"].agg(["mean", "count"])
by_fam = by_fam[by_fam["count"] >= 5]          # ignore sizes with too few people
alone = fam.assign(Alone=fam.FamilySize == 1).groupby("Alone")["Survived"].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2),
                               gridspec_kw={"width_ratios": [1.4, 1]})

bars = ax1.bar(range(len(by_fam)), by_fam["mean"], color=SURVIVED,
               width=0.6, edgecolor=SURFACE, linewidth=2, zorder=3)
pct_labels(ax1, bars, by_fam["mean"], size=10)
ax1.set_xticks(range(len(by_fam)),
               [f"{s}\nn={n}" for s, n in zip(by_fam.index.astype(str), by_fam["count"])])
ax1.set_ylim(0, 0.85)
ax1.set_xlabel("Family size aboard (self + siblings/spouse + parents/children)")
ax1.set_ylabel("Survival rate")
ax1.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax1.set_title("Small families did best; large ones did worst")
tidy(ax1)

bars = ax2.bar(["Alone", "With family"], [alone[True], alone[False]],
               color=SURVIVED, width=0.45, edgecolor=SURFACE, linewidth=2, zorder=3)
pct_labels(ax2, bars, [alone[True], alone[False]], size=13)
ax2.set_ylim(0, 0.72)
ax2.set_ylabel("Survival rate")
ax2.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax2.set_title("Travelling alone was worse")
tidy(ax2)

plt.tight_layout()
plt.show()

print("SibSp and Parch are weak alone, but their sum (family size) is a better feature.")

In [ ]:
# ---- Chart 7: port of embarkation -------------------------------------------
port_names = {"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"}
emb = train_data.dropna(subset=["Embarked"])
by_port = emb.groupby("Embarked")["Survived"].agg(["mean", "count"]).sort_values("mean")
class_mix = pd.crosstab(emb["Embarked"], emb["Pclass"], normalize="index").loc[by_port.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))

names = [f"{port_names[p]}\n(n={c})" for p, c in zip(by_port.index, by_port["count"])]
bars = ax1.bar(names, by_port["mean"], color=SURVIVED, width=0.55,
               edgecolor=SURFACE, linewidth=2, zorder=3)
pct_labels(ax1, bars, by_port["mean"], size=12)
ax1.set_ylim(0, 0.72)
ax1.set_ylabel("Survival rate")
ax1.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax1.set_title("Cherbourg passengers fared best")
tidy(ax1)

bottom = np.zeros(len(class_mix))
shades = ["#1c5cab", "#3987e5", "#9ec5f4"]
for i, c in enumerate([1, 2, 3]):
    vals = class_mix[c].values
    ax2.bar(range(len(class_mix)), vals, bottom=bottom, width=0.55,
            color=shades[i], label=f"{c}{'st' if c==1 else 'nd' if c==2 else 'rd'} class",
            edgecolor=SURFACE, linewidth=2, zorder=3)
    bottom += vals
ax2.set_xticks(range(len(class_mix)), [port_names[p] for p in class_mix.index])
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax2.set_ylabel("Share of passengers")
ax2.set_title("...because it boarded the most 1st-class passengers")
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
tidy(ax2)

plt.tight_layout()
plt.show()

print("Port looks predictive, but it is mostly a proxy for class.")

In [ ]:
# ---- Chart 8: titles hidden inside the Name column --------------------------
titles = train_data["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
common = ["Mr", "Mrs", "Miss", "Master"]
title_group = titles.where(titles.isin(common), "Other")
by_title = train_data.groupby(title_group)["Survived"].agg(["mean", "count"])
by_title = by_title.reindex(["Mrs", "Miss", "Master", "Other", "Mr"])

fig, ax = plt.subplots(figsize=(9.5, 4))
bars = ax.bar(range(len(by_title)), by_title["mean"], color=SURVIVED,
              width=0.55, edgecolor=SURFACE, linewidth=2, zorder=3)
pct_labels(ax, bars, by_title["mean"], size=12)
ax.set_xticks(range(len(by_title)),
              [f"{t}\nn={n}" for t, n in zip(by_title.index.astype(str), by_title["count"])])
ax.axhline(overall, color=MUTED, linewidth=1, zorder=2)
ax.text(len(by_title) - 0.45, overall + 0.018, f"average {overall:.1%}",
        color=MUTED, fontsize=9, ha="right")
ax.set_ylim(0, 0.92)
ax.set_ylabel("Survival rate")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("'Master' (a boy) survived far more often than 'Mr'")
tidy(ax)
plt.tight_layout()
plt.show()

print("Title is free information sitting in the Name column: it encodes sex, "
      "rough age and status at once, and is one of the easiest ways to improve this model.")

In [ ]:
# ---- Chart 9: how the numeric features relate to each other -----------------
corr_df = train_data.copy()
corr_df["IsFemale"] = (corr_df["Sex"] == "female").astype(int)
corr_df["FamilySize"] = corr_df["SibSp"] + corr_df["Parch"] + 1
cols = ["Survived", "IsFemale", "Pclass", "Fare", "Age", "FamilySize", "SibSp", "Parch"]
corr = corr_df[cols].corr()

# Hide the mirror half and the all-1.00 diagonal: they carry no information.
shown = np.tril(np.ones_like(corr.values, dtype=bool), k=-1)
masked = np.where(shown, corr.values, np.nan)

fig, ax = plt.subplots(figsize=(7.6, 6.2))
im = ax.imshow(masked, cmap=DIVERGING, vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right")
ax.set_yticks(range(len(cols)), cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        if not shown[i, j]:
            continue
        v = corr.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9.5,
                fontweight="bold" if abs(v) > 0.3 else "normal",
                color="#ffffff" if abs(v) > 0.55 else INK)
ax.set_title("Correlation with survival: sex dominates")
ax.grid(False)
cell_gaps(ax, len(cols), len(cols))
for s in ax.spines.values():
    s.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
cb.ax.tick_params(colors=MUTED, labelcolor=SECOND, length=0)
cb.outline.set_visible(False)
plt.tight_layout()
plt.show()

print("Correlations with Survived, strongest first:")
print(corr["Survived"].drop("Survived").abs().sort_values(ascending=False).round(3).to_string())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

TITLE_MAP = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
KEEP = ["Mr", "Mrs", "Miss", "Master"]

def build_features(d):
    x = pd.DataFrame(index=d.index)
    x["Pclass"] = d["Pclass"]
    x["Sex"] = d["Sex"]
    x["Age"] = d["Age"]                       # imputed in the pipeline
    x["Fare"] = d["Fare"]                     # imputed in the pipeline
    fam = d["SibSp"] + d["Parch"] + 1
    x["FamilySize"] = fam
    x["IsAlone"] = (fam == 1).astype(int)
    # "Braund, Mr. Owen Harris" -> "Mr"
    title = d["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    title = title.replace(TITLE_MAP)
    x["Title"] = np.where(title.isin(KEEP), title, "Rare")
    return x

X      = build_features(train_data)
X_test = build_features(test_data)
y      = train_data["Survived"]

CAT = ["Sex", "Title"]
NUM = ["Pclass", "Age", "Fare", "FamilySize", "IsAlone"]

# Imputation lives inside the pipeline, so medians are learned from the
# training folds only -- no leakage into the validation fold.
model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ])),
    ("clf", GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=3, random_state=1)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print(f"5-fold CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({"PassengerId": test_data.PassengerId, "Survived": predictions})
output.to_csv("submission.csv", index=False)
print("Your submission was successfully saved!")

### Model evaluation

The submitted model is a **gradient-boosting classifier** on engineered features:
`Title` pulled out of `Name`, imputed `Age` and `Fare`, `FamilySize` and `IsAlone`
derived from `SibSp`/`Parch`, alongside `Pclass` and `Sex`. It replaces the first
submission, a random forest on four raw columns.

Both models are scored on the **same five folds** below, so the comparison is fair.
The confusion matrix then shows *which* passengers the new model still gets wrong.

In [ ]:
# ---- Chart 10: did the new model actually beat the old one? -----------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict

# The old submission: a random forest on four raw columns.
old_X = pd.get_dummies(train_data[["Pclass", "Sex", "SibSp", "Parch"]])
old_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)

# `model`, `X`, `y` and `cv` come from the modelling cell above (gradient
# boosting on the engineered features). Both are scored on the same folds.
old_scores = cross_val_score(old_model, old_X, y, cv=cv, scoring="accuracy")
new_scores = cross_val_score(model,     X,     y, cv=cv, scoring="accuracy")

preds = cross_val_predict(model, X, y, cv=cv)
cm = pd.crosstab(y, preds).values
labels = ["Did not survive", "Survived"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3),
                               gridspec_kw={"width_ratios": [1.25, 1]})

# --- Left: per-fold accuracy, old vs new -------------------------------------
idx = np.arange(len(new_scores))
w = 0.38
b1 = ax1.bar(idx - w/2, old_scores, width=w, color=BASELINE,
             edgecolor=SURFACE, linewidth=1.5, zorder=3,
             label="Old: random forest, 4 raw columns")
b2 = ax1.bar(idx + w/2, new_scores, width=w, color=SURVIVED,
             edgecolor=SURFACE, linewidth=1.5, zorder=3,
             label="New: gradient boosting + features")
ax1.bar_label(b1, labels=[f"{s:.3f}" for s in old_scores], padding=3,
              color=SECOND, fontsize=8.5)
ax1.bar_label(b2, labels=[f"{s:.3f}" for s in new_scores], padding=3,
              color=INK, fontsize=8.5, fontweight="bold")
ax1.axhline(old_scores.mean(), color=BASELINE, linewidth=1.2, linestyle="--", zorder=2)
ax1.axhline(new_scores.mean(), color=SURVIVED, linewidth=1.2, linestyle="--", zorder=2)
ax1.set_xticks(idx, [f"Fold {i+1}" for i in idx])
ax1.set_ylim(0.60, 0.95)
ax1.set_ylabel("Accuracy")
ax1.set_title(f"Accuracy per fold: {old_scores.mean():.3f} -> {new_scores.mean():.3f} "
              f"({new_scores.mean() - old_scores.mean():+.3f})")
ax1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=2, fontsize=8.5)
tidy(ax1)

# --- Right: confusion matrix of the NEW model --------------------------------
ax2.imshow(cm, cmap=BLUES, vmin=0, vmax=cm.max())
ax2.set_xticks(range(2), labels)
ax2.set_yticks(range(2), labels)
ax2.set_xlabel("Predicted")
ax2.set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        ax2.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=17,
                 fontweight="bold", color="#ffffff" if cm[i, j] > cm.max() * 0.55 else INK)
ax2.set_title("Confusion matrix, new model")
ax2.grid(False)
cell_gaps(ax2, 2, 2)
for s in ax2.spines.values():
    s.set_visible(False)

plt.tight_layout()
plt.show()

print(f"Old model (RF, 4 raw columns):        {old_scores.mean():.4f} (+/- {old_scores.std():.4f})")
print(f"New model (GB, engineered features):  {new_scores.mean():.4f} (+/- {new_scores.std():.4f})")
print(f"Improvement:                          {new_scores.mean() - old_scores.mean():+.4f}\n")
print(f"Missed survivors (predicted dead, actually lived): {cm[1, 0]}")
print(f"False alarms  (predicted alive, actually died):    {cm[0, 1]}")

In [ ]:
# ---- Chart 11: what the new model actually leans on -------------------------
# `model` is a Pipeline, so the importances live on the classifier step and the
# column names come from the fitted preprocessor.
model.fit(X, y)
names = model.named_steps["prep"].get_feature_names_out()
names = [n.split("__", 1)[1] for n in names]          # drop the "num__"/"cat__" prefix
imp = pd.Series(model.named_steps["clf"].feature_importances_, index=names).sort_values()

colors = [SURVIVED if v >= 0.10 else BASELINE for v in imp.values]

fig, ax = plt.subplots(figsize=(8.5, 4.4))
bars = ax.barh(imp.index, imp.values, color=colors, height=0.62,
               edgecolor=SURFACE, linewidth=1.5, zorder=3)
ax.bar_label(bars, labels=[f"{v:.2f}" for v in imp.values], padding=5,
             color=INK, fontsize=10, fontweight="bold")
ax.set_xlim(0, max(imp.values) * 1.18)
ax.set_xlabel("Feature importance")
ax.set_title("Title and fare now carry weight the old model never saw")
tidy(ax, grid_axis="x")
plt.tight_layout()
plt.show()

print(imp.sort_values(ascending=False).round(3).to_string())
print("\nThe old model could only rank Sex, Pclass, SibSp and Parch. The new one spreads")
print("its weight across Title, Fare and Age - information that was in the data all along.")

### Where the improvement actually came from

A better score is only interesting if you know what produced it. Running four
configurations over the same folds separates the two possible explanations --
a stronger algorithm, or better inputs.

In [ ]:
# ---- Chart 12: where the improvement actually came from ---------------------
# Four configurations, all scored on the same 5 folds, to separate
# "better model" from "better features".
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

def gb():
    return GradientBoostingClassifier(n_estimators=300, learning_rate=0.05,
                                      max_depth=3, random_state=1)

def pipe(estimator, cat, num):
    return Pipeline([("prep", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])),
        ("clf", estimator)])

raw_X = train_data[["Pclass", "Sex", "SibSp", "Parch"]]

configs = {
    "Guess: all women live,\nall men die":      ((train_data.Sex == "female").astype(int) == y).mean(),
    "Old: random forest,\n4 raw columns":       cross_val_score(pipe(RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1), ["Sex"], ["Pclass", "SibSp", "Parch"]), raw_X, y, cv=cv).mean(),
    "Gradient boosting,\nsame 4 raw columns":   cross_val_score(pipe(gb(), ["Sex"], ["Pclass", "SibSp", "Parch"]), raw_X, y, cv=cv).mean(),
    "New: gradient boosting,\n+ engineered features": cross_val_score(pipe(gb(), CAT, NUM), X, y, cv=cv).mean(),
}

names = list(configs)
vals = list(configs.values())
colors = [BASELINE, BASELINE, BASELINE, SURVIVED]

fig, ax = plt.subplots(figsize=(10, 4.6))
bars = ax.bar(names, vals, color=colors, width=0.6,
              edgecolor=SURFACE, linewidth=2, zorder=3)
ax.bar_label(bars, labels=[f"{v:.3f}" for v in vals], padding=5,
             color=INK, fontsize=11, fontweight="bold")
ax.set_ylim(0.70, 0.90)
ax.set_ylabel("5-fold CV accuracy")
ax.set_title("Changing the model did nothing. Adding features did the work.")
ax.tick_params(axis="x", labelsize=9)
tidy(ax)

# Label the two steps that matter: the model swap, then the feature work.
def step(i, j, text, color):
    top = max(vals[i], vals[j])
    ax.annotate("", xy=(j, top + 0.019), xytext=(i, top + 0.019),
                arrowprops=dict(arrowstyle="->", color=color, linewidth=1.4))
    ax.text((i + j) / 2, top + 0.024, text, ha="center", color=color,
            fontsize=10, fontweight="bold")

step(1, 2, f"swap the model: {vals[2] - vals[1]:+.3f}", MUTED)
step(2, 3, f"add features: {vals[3] - vals[2]:+.3f}", SURVIVED)

plt.tight_layout()
plt.show()

print(f"Swapping RF -> gradient boosting on the same 4 columns: {vals[2] - vals[1]:+.4f}")
print(f"Adding Title / Age / Fare / FamilySize / IsAlone:       {vals[3] - vals[2]:+.4f}")

## Reflection and plan for improvement

**Result.** The first submission (random forest, `Pclass` / `Sex` / `SibSp` / `Parch`)
cross-validated at about **0.795**. Replacing it with gradient boosting on engineered
features moved that to about **0.848** -- roughly **+5 accuracy points**, or about a
third of the gap between the naive "all women survive" guess and a perfect score.

| | Kaggle public score |  |
|---|---|---|
| First submission (random forest) | 0.77511 V3 |  |
| Current submission (gradient boosting) | 0.76315 V73 |  |

**What I learned.** My instinct was that a stronger model would give a better score.
The experiment above says otherwise: swapping the random forest for gradient boosting
on the *same four columns* changed nothing (it was actually slightly worse, and well
inside the fold-to-fold noise of ±0.02). Every other algorithm I tried on those four
columns -- logistic regression, SVM, extra trees -- landed in the same narrow band.
Four raw columns simply do not contain more signal than that, and no algorithm can
extract information that is not there.

The entire improvement came from **feature engineering**. `Title`, extracted from a
text column the first model ignored completely, encodes sex, rough age and social
status in one variable. `Fare` turned out to be the single most valuable addition --
removing it costs more accuracy than removing any other feature. Collapsing `SibSp`
and `Parch` into `FamilySize` gave the model a cleaner version of the same
information. I also moved missing-value imputation *inside* the pipeline, so fold
medians are learned from training data only; doing it beforehand would have leaked
validation data into training and inflated the score I was reporting.

**Expect the leaderboard to read lower than cross-validation.** The public
leaderboard scores only 418 passengers, so a 0.848 CV estimate realistically lands
around 0.78-0.80 there. That gap is sample size, not overfitting.

**Ideas for the next iteration**

1. **Impute `Age` by `Title`** rather than a global median -- a missing "Master" is a
   child, a missing "Mr" is not, and the current median erases that distinction.
2. **Group features from `Ticket` and `Cabin`.** Passengers sharing a ticket number
   travelled together and tended to live or die together; `Cabin`'s first letter
   encodes deck, which is a proxy for how far you were from a lifeboat.
3. **Tune hyperparameters properly** with `GridSearchCV` inside a nested
   cross-validation loop, so the tuning itself does not leak into the reported score.
4. **Ensemble** the gradient booster with logistic regression and an SVM through
   `VotingClassifier` -- they make different mistakes, so averaging them often helps.
5. **Know when to stop.** Public leaderboard scores much above ~0.82 usually mean
   someone looked up the real passenger manifest, which is not modelling. Past that
   point, differences between honest submissions are mostly noise.